# conv-channel-sum — worked example 1: Reproduce F.conv2d with an explicit einsum over IC

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-channel-sum`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`F.conv2d` slides each output-channel filter over the input and, at every spatial position, multiplies the `(IC, KH, KW)` patch by the `(IC, KH, KW)` kernel and **sums all of it** into a single scalar. The IC axis is *contracted* — it disappears from the output. If we unfold the input into sliding patches, the whole convolution becomes one `einsum` whose summed (repeated) indices are exactly `IC, KH, KW`.

## Worked solution

We want to show the channel-sum semantics by replacing the conv with one `einsum`.

**Step 1 — unfold the input into patches.** `F.unfold(x, (KH, KW))` turns `x: (B, IC, H, W)` into `(B, IC*KH*KW, L)` where `L = OH*OW` is the number of sliding window positions. Reshaping to `(B, IC, KH, KW, OH, OW)` recovers a named-axis view of every patch. This works because unfold lays out the receptive field contents contiguously per position, so the IC/KH/KW block can be split back out.

**Step 2 — name the kernel axes.** `weight` is `(OC, IC, KH, KW)`. No reshape is needed; we just feed it to einsum.

**Step 3 — write the contraction.** The einsum `'bckhpq,ockh->bopq'` multiplies patches `(b, ic=c, kh=k, kw=h, oh=p, ow=q)` against kernel `(oc=o, ic=c, kh=k, kw=h)` and sums over every index that appears in both inputs but not the output: `c (IC)`, `k (KH)`, `h (KW)`. Those three are the contracted axes — precisely the channel-and-kernel sum that defines a convolution.

**Step 4 — verify.** Because einsum sums exactly the IC, KH, KW axes, the result matches `F.conv2d(x, weight)` to floating-point tolerance. Seeing `c` appear on both factors and vanish from the output is the visual proof that conv2d sums over in-channels.

In [ ]:
import torch.nn.functional as F

def conv2d_via_einsum(x, weight):
    B, IC, H, W = x.shape
    OC, _, KH, KW = weight.shape
    OH, OW = H - KH + 1, W - KW + 1
    cols = F.unfold(x, (KH, KW))                       # (B, IC*KH*KW, OH*OW)
    patches = cols.reshape(B, IC, KH, KW, OH, OW)      # name the axes
    # contract over c=IC, k=KH, h=KW
    y = t.einsum('bckhpq,ockh->bopq', patches, weight)
    return y

t.manual_seed(0)
x = t.randn(2, 3, 7, 6)
weight = t.randn(4, 3, 3, 2)
y = conv2d_via_einsum(x, weight)
ref = F.conv2d(x, weight)
print(y.shape, t.allclose(y, ref, atol=1e-4))